# Importing Libraries


In [ ]:
import cv2
import mediapipe as mp


import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import random
import os
from tqdm.notebook import tqdm

import IPython.display as ipd

TypeError: Descriptors cannot be created directly.
If this call came from a _pb2.py file, your generated code is out of date and must be regenerated with protoc >= 3.19.0.
If you cannot immediately regenerate your protos, some other possible workarounds are:
 1. Downgrade the protobuf package to 3.20.x or lower.
 2. Set PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python (but this will use pure-Python parsing and will be much slower).

More information: https://developers.google.com/protocol-buffers/docs/news/2022-05-06#python-updates

# Extracting Required Paths


In [ ]:
glob_path = [i for i in glob('Dataset\*\*\*.mov')]
output_label = [i for i in glob('Dataset\*\*')]

glob_path,output_label

: 

In [ ]:
input_path = []

for i in range(0, len(train_data)):
    input_path.append('Dataset/' + train_data['video_path'][i])


## Vedio Augmentaion

1. Centre Crop
2. Flip
3. Org  
4. Upsample
5. Downsample

### Optional (Optional Experimentrs)
6. sharpen 
7. blurred
8. Frame Skipping

# Loading Video in OpenCV

In [ ]:
random_video = random.choice(input_path)
print(random_video.split("/")[-2])

cap = cv2.VideoCapture(random_video)
print(f"Video Resolution: {cap.get(cv2.CAP_PROP_FRAME_WIDTH)} x {cap.get(cv2.CAP_PROP_FRAME_HEIGHT)}")
print(f"FPS: {cap.get(cv2.CAP_PROP_FPS)}")

ipd.Video(random_video,width=700)

# Prepering Mediapipe

In [30]:
mp_holistic = mp.solutions.holistic # Holistic model
mp_drawing = mp.solutions.drawing_utils # Drawing utilities

In [31]:
def mediapipe_detection(image, model) -> tuple:
    image_ = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) 
    image_.flags.writeable = False                  
    results = model.process(image_) 
    image_.flags.writeable = True                   
    image = cv2.cvtColor(image_, cv2.COLOR_RGB2BGR) 
    return image, results

In [32]:
def draw_landmarks(image, results) -> None:
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS) # Draw pose connections
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS) # Draw left hand connections
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS) # Draw right hand connections

In [33]:
def draw_styled_landmarks(image, results) -> None:
    # Draw pose connections
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
                             ) 
    
    # Draw left hand connections
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                             mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)
                             ) 
    
    # Draw right hand connections  
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                             mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
                             ) 

In [34]:
def extract_keypoints(results) -> np.array:
    
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*4)
    face = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(468*3)
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    
    return np.concatenate([pose, face, lh, rh])

# Collecting MP data


## INCLUDE 50


In [ ]:

input_path_I50 = []
for i in range(len(train_data)):
    if train_data['include_50'][i] == True:
        input_path_I50.append("Dataset\\" + train_data['video_path'][i])
        

input_path_I50= pd.Series(input_path_I50)

input_path_I50.sample(10)


In [ ]:

output_label_I50 = []
for i in range(len(train_data)):
    if train_data['include_50'][i] == True:
        output_label_I50.append(train_data['label'][i])

output_label_I50= pd.Series(output_label_I50)

output_label_I50.sample(10)


In [ ]:
output_label_dict_I50 = {}

for i in range(len(output_label_I50)):
    output_label_dict_I50[output_label_I50[i]] = output_label_dict_I50.get(output_label_I50[i],0)+1 

output_label_dict_I50

In [ ]:
random_video = random.choice(input_path_I50)
cap = cv2.VideoCapture(random_video)

current_label = random_video.split("/")[-2]
current_label = current_label.split(" ")[1:]
current_label = " ".join(current_label)


# Actions that we try to detect
actions = np.array(list(output_label_dict_I50.keys()))

no_sequences = output_label_dict_I50[current_label]
vedio_length = cap.get(cv2.CAP_PROP_FRAME_COUNT)

print(current_label.title(), f"with {no_sequences} samples")
print(f"Max Frames: {vedio_length}")
print(f"Video Resolution: {cap.get(cv2.CAP_PROP_FRAME_WIDTH)} x {cap.get(cv2.CAP_PROP_FRAME_HEIGHT)}")
print(f"FPS: {cap.get(cv2.CAP_PROP_FPS)}")

ipd.Video(random_video,width=700)

In [39]:
#Creating dir for each label
try:
    for i in range(len(output_label_dict_I50)):
        os.mkdir(f'MP_data/{output_label_dict_I50[i]}')
except:
    pass

In [ ]:
def display_cv2_image(img, figsize = (10,10)):
    img_ = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    fig,ax = plt.subplots(figsize=figsize)
    ax.imshow(img_)
    ax.axis('off')
    
ret, frame = cap.read()

img,res = mediapipe_detection(frame, mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5))

cap.release()

draw_landmarks(img, res)

display_cv2_image(img)



In [ ]:
input_path_I50.sample(10)

In [ ]:
holistic = mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5)
DATA_PATH = "MP_data/"
    
# Loop through sequences aka videos
# for input_path in tqdm(input_path_I50):
for input_path in tqdm(input_path_I50[:1]):
    cap = cv2.VideoCapture(input_path)
    all_keypoints = []
    keypoints = np.array(None)
        
    while cap.isOpened():           
        # Read feed
        ret, frame = cap.read()
        if not ret:
            break
        
        # Make detections
        image, results = mediapipe_detection(frame, holistic)

        # Draw landmarks
        # draw_styled_landmarks(image, results)
            
        #Collecting 3D landmarks for a single frame
        keypoints = extract_keypoints(results)
        all_keypoints.append(keypoints)        
    
    all_keypoints = np.vstack(all_keypoints)
    
    # Saving Landmarks
    path = input_path.split("/")
    
    # Getting Current action of the video
    action = path[-2]
    action = action.split(" ")[1:]
    action = " ".join(action)
    
    #Get the sequence number
    sequence = path[-1]
    sequence = sequence.split(".")[0]
    
    npy_path = os.path.join(DATA_PATH, action, sequence)
    print(npy_path)
    
    np.save(npy_path, all_keypoints)

    cap.release()